# BNC state observability on wrist-camera validation

This targeted review pairs correct predictions and INSERT/REMOVE confusions, including failed attempts. Each selected window comes from a different annotated segment. The original annotations identify cases for inspection. Read visual states from the images before interpreting the action outcome.

The gray context panels use frames outside the model input. They help locate what happened before or after the prediction window. The sensor plots share the wrist-camera clock through the original HDF5 timestamps.

In [ ]:
import json
from pathlib import Path

import cv2
import h5py
import matplotlib.pyplot as plt
import numpy as np

repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "main.py").is_file())
run_dir = repo / "outputs/experiments/reassemble_hand_finetune"
run = json.loads((run_dir / "run.json").read_text(encoding="utf-8"))
plan = json.loads((repo / "outputs/visual_reassemble_hand/plan.json").read_text(encoding="utf-8"))
raw_dir = Path(plan["dataset_plan"]["raw_dir"])
predictions = {
    row["clip_id"]: row
    for row in json.loads((run_dir / "validation_predictions.json").read_text(encoding="utf-8"))
}

clip_ids = [
    "2025-01-09-14-57-02_hand_clip_0021",  # Correct INSERT
    "2025-01-13-15-18-12_hand_clip_0024",  # Correct REMOVE
    "2025-01-09-18-08-36_hand_clip_0146",  # Successful INSERT predicted as REMOVE
    "2025-01-09-19-15-33_hand_clip_0285",  # Successful INSERT predicted as REMOVE
    "2025-01-13-11-01-25_hand_clip_0157",  # Successful REMOVE predicted as INSERT
    "2025-01-14-12-05-01_hand_clip_0058",  # Successful REMOVE predicted as INSERT
    "2025-01-09-19-15-33_hand_clip_0262",  # Failed INSERT predicted as REMOVE
    "2025-01-09-14-40-21_hand_clip_0170",  # Failed REMOVE predicted as INSERT
]
clips = [predictions[clip_id] for clip_id in clip_ids]
assert set(clip_ids) <= {clip["clip_id"] for clip in run["validation_clips"]}
assert len({(clip["recording_id"], clip["segment_id"]) for clip in clips}) == len(clips)
assert all(clip["camera"] == "hand" and clip["object"] == "BNC" for clip in clips)

for clip in clips:
    print(
        clip["clip_id"], "| segment", clip["segment_id"],
        "|", clip["ground_truth"], "→", clip["predicted_activity"],
        "| success:", clip["success"],
        "| MP4:", f'{clip["start_time"]:.1f}–{clip["end_time"]:.1f} s',
        "| score:", round(clip["candidates"][0]["score"], 3),
    )

In [ ]:
def h5_path(clip):
    return raw_dir / f'{clip["recording_id"]}.h5'


def frame_sheet(clip):
    video_path = run["videos"][clip["recording_id"]]["path"]
    first, stop = clip["start_frame"], clip["end_frame_exclusive"]
    t0 = clip["original_start_time"]
    with h5py.File(h5_path(clip), "r") as recording:
        camera_t = recording["timestamps/hand"][:].reshape(-1)
        segment = recording["segments_info"][clip["segment_id"]]
        segment_start, segment_end = float(segment["start"][()]), float(segment["end"][()])
    assert np.isclose(camera_t[first], t0, rtol=0, atol=1e-6)
    assert np.isclose(camera_t[stop], clip["original_end_time_exclusive"], rtol=0, atol=1e-6)
    frames = np.linspace(first, stop - 1, 8, dtype=int).tolist()
    titles = ["Window"] * len(frames)
    context = [
        ("Before window", camera_t[first] - 1.2),
        ("After window", camera_t[stop] + 1.2),
        ("Segment start", segment_start + 0.2),
        ("Segment end", segment_end - 0.2),
    ]
    for title, timestamp in context:
        frames.append(min(np.searchsorted(camera_t, timestamp), len(camera_t) - 1))
        titles.append(title)

    fig, axes = plt.subplots(3, 4, figsize=(13, 7.4), layout="constrained")
    fig.suptitle(clip["clip_id"])
    cap = cv2.VideoCapture(video_path)
    try:
        for ax, frame_index, title in zip(axes.flat, frames, titles):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
            ok, frame = cap.read()
            if not ok:
                raise RuntimeError(f"Cannot decode frame {frame_index} from {video_path}")
            ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            ax.set_title(f"{title}  {camera_t[frame_index] - t0:+.2f} s", fontsize=9)
            if title != "Window":
                ax.title.set_color("dimgray")
            ax.axis("off")
    finally:
        cap.release()
    return fig

for clip in clips:
    frame_sheet(clip)
    plt.show()

In [ ]:
def sensor_window(recording, key, start, end):
    # Slice by the sensor's own timestamps, on the same clock as the camera.
    time = recording[f"timestamps/{key}"][:].reshape(-1)
    samples = recording[f"robot_state/{key}"]
    if len(time) != len(samples) or not np.isfinite(time).all() or np.any(np.diff(time) < 0):
        raise ValueError(f"Invalid timestamps for {key}")
    left, right = np.searchsorted(time, [start, end])
    if left == right:
        raise ValueError(f"No samples for {key} in this interval")
    values = samples[left:right]
    if not np.isfinite(values).all():
        raise ValueError(f"Invalid samples for {key}")
    return time[left:right], values


def telemetry_plot(clip):
    t0 = clip["original_start_time"]
    duration = clip["original_end_time_exclusive"] - t0
    keys = ("pose", "gripper_positions", "measured_force", "measured_torque")
    with h5py.File(h5_path(clip), "r") as recording:
        data = {
            key: sensor_window(recording, key, t0 - 2.5, t0 + duration + 2.5)
            for key in keys
        }

    pose_t, pose = data["pose"]
    jaw_t, jaws = data["gripper_positions"]
    force_t, force = data["measured_force"]
    torque_t, torque = data["measured_torque"]

    # Relative rotation magnitude avoids assuming a common board axis.
    quaternions = pose[:, 3:]
    quaternions = quaternions / np.linalg.norm(quaternions, axis=1, keepdims=True)
    rotation = np.degrees(2 * np.arccos(np.clip(np.abs(quaternions @ quaternions[0]), 0, 1)))

    fig, axes = plt.subplots(4, 1, figsize=(9, 7.4), sharex=True, layout="constrained")
    fig.suptitle(clip["clip_id"])
    axes[0].plot(pose_t - t0, 1000 * (pose[:, :3] - pose[0, :3]), label=("x", "y", "z"))
    axes[0].set_ylabel("EE delta (mm)")
    rotation_axis = axes[0].twinx()
    rotation_axis.plot(pose_t - t0, rotation, color="gray", alpha=0.6)
    rotation_axis.set_ylabel("Rotation (deg)")
    axes[1].plot(jaw_t - t0, 1000 * jaws, label=("left", "right"))
    axes[1].set_ylabel("Fingers (mm)")
    axes[2].plot(force_t - t0, np.linalg.norm(force, axis=1), color="tab:green")
    axes[2].set_ylabel("Force norm (N)")
    axes[3].plot(torque_t - t0, np.linalg.norm(torque, axis=1), color="tab:purple")
    axes[3].set_ylabel("Torque norm (N m)")
    for ax in axes:
        ax.axvspan(0, duration, color="0.8", alpha=0.35)
        ax.grid(alpha=0.25)
    axes[0].legend(loc="upper left", ncol=3)
    axes[1].legend(loc="upper left", ncol=2)
    axes[-1].set_xlabel("Seconds from window start")
    return fig

for clip in clips:
    telemetry_plot(clip)
    plt.show()

### Frames near the end of each successful attempt

Times are relative to the end of the model window. Negative times are inside the window. The center crop makes it easier to compare the moving connector with the socket fixed to the board.

In [ ]:
offsets = [-0.4, -0.06, 0.2, 0.45, 0.7, 1.05, 1.6, 2.2]
successful_clips = [clip for clip in clips if clip["success"]]
fig, axes = plt.subplots(len(successful_clips), len(offsets), figsize=(15, 9), layout="constrained")
fig.suptitle("BNC at the window boundary")
for clip, row in zip(successful_clips, axes):
    stop_time = clip["original_end_time_exclusive"]
    with h5py.File(h5_path(clip), "r") as recording:
        camera_t = recording["timestamps/hand"][:].reshape(-1)
    cap = cv2.VideoCapture(run["videos"][clip["recording_id"]]["path"])
    try:
        for ax, index in zip(row, np.searchsorted(camera_t, stop_time + np.array(offsets))):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(index))
            ok, frame = cap.read()
            if not ok:
                raise RuntimeError(f'Cannot decode frame {index} for {clip["clip_id"]}')
            h, w = frame.shape[:2]
            crop = frame[round(h * 0.20):round(h * 0.84), round(w * 0.26):round(w * 0.74)]
            ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
            ax.set_title(f'{clip["clip_id"][-4:]}  {camera_t[index] - stop_time:+.2f} s', fontsize=8)
            ax.axis("off")
    finally:
        cap.release()

plt.show()

### Manual observations

The table distinguishes what is visible at the window endpoints from what becomes clear in the surrounding frames. Seating inferred from the piece staying on the board after release is visual evidence of engagement. The locking mechanism itself is small and partly occluded.

| Clip suffix | Annotation and prediction | Observation at window start | Observation at window end and in context |
|---|---|---|---|
| 0021 | Successful INSERT predicted as INSERT | Held against the socket, lock uncertain | The jaws open within the window and the connector stays in the socket as the wrist moves away. |
| 0024 | Successful REMOVE predicted as REMOVE | Held near the socket, joint partly hidden | Wrist retreat begins around the boundary with the jaws closed. The wrist view makes the instant of separation hard to place. |
| 0146 | Successful INSERT predicted as REMOVE | Held at the socket, lock uncertain | Still held at the boundary. Release is visible in the late segment context about 5.6 s after the window. |
| 0285 | Successful INSERT predicted as REMOVE | Held at the socket, lock uncertain | Still held at the boundary. The piece stays at the socket after the jaws open in the following frames, about 1.6 s after the window. |
| 0157 | Successful REMOVE predicted as INSERT | Held at the socket, lock uncertain | The connector and the fixed socket still meet in the final window frames. A visible gap appears in the following context, about 1.1 s after the window. |
| 0058 | Successful REMOVE predicted as INSERT | Held at the socket, lock uncertain | The connector is still close to the socket at the boundary. It is visibly separated and still gripped about 0.7 s after the window. |
| 0262 | Failed INSERT predicted as REMOVE | Held near the socket | The jaws stay closed and the wrist turns. The sampled window gives an uncertain engagement state. |
| 0170 | Failed REMOVE predicted as INSERT | Held near the socket | The jaws stay closed while the wrist turns. At the late segment frame the jaws have opened and the connector appears to remain at the socket. No separation is evident in the sampled frames. |

In the clear successful INSERT examples, jaw opening accompanies withdrawal while the connector stays at the socket. In the clear REMOVE examples, the wrist withdraws with the jaws closed while the connector moves with it. Torque magnitude varies in both directions and in failed attempts, so a torque peak alone leaves the action ambiguous. These eight purposively chosen windows illustrate possible cues and their timing. They do not estimate their frequency or the effect on accuracy.